In [ ]:
"""
Boussinesq: rho = rho-f sauf dans le terme gravité
  Aquifère :  div(Kf grad h) = -div(Kf buoy e_z)         
  Mer      :  h forcé vers h_ref(z) = (rho_s-rho_f)/rho_f * (0 - z)
  q = -Kf_face (grad h + buoy e_z), mêmes coefficients partout
  Transport : ne dC/dt + div(q C) - div(D grad C) = 0, upwind, C=35
              réimposé dans la mer.
"""

from fipy import Grid2D, CellVariable, FaceVariable, DiffusionTerm
from fipy import TransientTerm, UpwindConvectionTerm, ImplicitSourceTerm
from fipy.solvers.scipy import LinearLUSolver
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import time as walltime

OUT = Path(r"C:\Users\Admin\frames_robinson_22")
try:
    OUT.mkdir(parents=True, exist_ok=True)
except OSError:
    OUT = Path("./frames_robinson2"); OUT.mkdir(parents=True, exist_ok=True)

if not (OUT / "checkpoint.pkl").exists():
    for f in OUT.glob("frame_*.png"):
        f.unlink()
print("VERSION v2 — h_ref hydrostatique, K_mer=1e4, eps=1e6, "
      "biseau initialisé côté mer")

solver = LinearLUSolver(tolerance=1e-12, iterations=2000)

#Paramètre
LL, LS   = 150.0, 50.0
L        = LL + LS
H        = 30.0
A        = 2.0                 # hauteur de plage émergée 
H_tot    = H + A
tan_beta = 0.1
Kf       = 10.0                # m/j
K_mer    = 1e4                 # assez grand : conductance d'interface
K_sec    = 1e-10               # plage émergée / air
ne_aq    = 0.25
rho_f    = 1000.0
drho_dC  = 0.7143
C_mer    = 35.0
C_terre  = 0.1                 
buoy_mer = drho_dC*C_mer/rho_f # = 0.025
Qf       = 2.1                 # 
alpha_L  = 0.5
alpha_T  = 0.05
Dm       = 1e-9 * 86400.0      # m2/j

Nx, Nz = 100, 64
dx, dz = L/Nx, H_tot/Nz

mesh = Grid2D(dx=dx, dy=dz, nx=Nx, ny=Nz)

x_cells = np.array(mesh.cellCenters[0]) - LL
z_cells = np.array(mesh.cellCenters[1]) - H
x_faces = np.array(mesh.faceCenters[0]) - LL
z_faces = np.array(mesh.faceCenters[1]) - H

faces_left   = np.array(mesh.facesLeft)
faces_bottom = np.array(mesh.facesBottom)
faces_top    = np.array(mesh.facesTop)
ext          = np.array(mesh.exteriorFaces)
face_normals = np.array(mesh.faceNormals)
face_cell_ids = np.array(mesh.faceCellIDs)
nF = mesh.numberOfFaces


#  (pente : z = -tan_beta * x, de (-20,+2) à (+20,-2))
z_pente = -tan_beta*x_cells
mask_mer = (z_cells <= 0) & (
    ((x_cells >= 0) & (x_cells <= A/tan_beta) & (z_cells > z_pente)) |
    ((x_cells > A/tan_beta) & (z_cells >= -A)))
mask_sec = ((z_cells > 0) & (x_cells >= -A/tan_beta) & (z_cells > z_pente))
mask_aq  = ~mask_mer & ~mask_sec

print(f"Grille {Nx}x{Nz} : mer={mask_mer.sum()}, "
      f"émergé/air={mask_sec.sum()}, aquifère={mask_aq.sum()}")


# Champs de propriétés
Kf_field = np.where(mask_mer, K_mer, np.where(mask_sec, K_sec, Kf))
Kf_var   = CellVariable(mesh=mesh, value=Kf_field)
Kf_face  = Kf_var.harmonicFaceValue
Kf_fv    = np.array(Kf_face)              

ne_field = np.where(mask_aq, ne_aq, 1.0)
ne_var   = CellVariable(mesh=mesh, value=ne_field)

# Pénalisation 
h_ref_field = np.where(mask_mer, buoy_mer*(0.0 - z_cells), 0.0)
eps_field   = np.where(mask_mer, 1e6, 0.0)
eps_var     = CellVariable(mesh=mesh, value=eps_field)

# Flottabilité masquée sur les frontières imperméables 
buoy_mask = np.where(ext & ~faces_top, 0.0, 1.0)
buoy_mask[faces_top] = 0.0     

# Variables et équations
C = CellVariable(name="C", mesh=mesh,
                 value=np.where(mask_mer, C_mer, C_terre), hasOld=True)
C.constrain(C_terre, mesh.facesLeft)

bloc_sel = (~mask_mer) & (~mask_sec) & (x_cells > 30) & (z_cells < -2)
C.setValue(np.where(bloc_sel, C_mer, C.value))
C.updateOld()

h = CellVariable(name="h", mesh=mesh, value=0.0)
# Flux d'eau douce imposé à gauche : q.n = -qf (entrant), faces verticales
qf = Qf / H_tot
gradh_left = FaceVariable(mesh=mesh, value=0.0)
gradh_left.setValue(-qf/Kf)
h.faceGrad.constrain(gradh_left, mesh.facesLeft)

rho_cell = CellVariable(mesh=mesh, value=rho_f)
rho_face = rho_cell.arithmeticFaceValue

b_var  = CellVariable(mesh=mesh, value=0.0)
D_face = FaceVariable(mesh=mesh, rank=2)
q_face = FaceVariable(mesh=mesh, rank=1)
q_face.setValue(np.zeros((2, nF)))
buoy_flux = FaceVariable(mesh=mesh, rank=1)

eq_h = (DiffusionTerm(coeff=Kf_face)
        - ImplicitSourceTerm(coeff=eps_var)) == b_var

eq_C = (TransientTerm(coeff=ne_var)
        + UpwindConvectionTerm(coeff=q_face)
        - DiffusionTerm(coeff=D_face))


# Interface mer/aquifère 

c1, c2 = face_cell_ids[0], face_cell_ids[1]
sea1 = mask_mer[c1]
sea2 = np.where(c2 >= 0, mask_mer[np.maximum(c2, 0)], sea1)
faces_interface = (sea1 ^ sea2) & ~ext
faces_sea_int   = sea1 & sea2            # faces entièrement dans la mer
# normale orientée de c1 vers c2 : signe pour "entrant dans l'aquifère"
sign_into_aq = np.where(sea1, +1.0, -1.0)   # si c1 est mer, n va mer->aquif

def solve_flow():
    """h + reconstruction cohérente de q. Retourne (qx, qz, buoy)."""
    rho_cell.setValue(rho_f + drho_dC*C.value)
    buoy = buoy_mask * (np.array(rho_face) - rho_f)/rho_f
    bf = np.zeros((2, nF)); bf[1] = Kf_fv*buoy
    buoy_flux.setValue(bf)
    b_var.setValue(-np.array(buoy_flux.divergence)      
                   - eps_field*h_ref_field)             # cible pénalisation
    eq_h.solve(var=h, solver=solver)
    h_fg = h.faceGrad.value
    qx = -Kf_fv*h_fg[0]
    qz = -Kf_fv*(h_fg[1] + buoy)
    # Faces INTÉRIEURES à la mer : C y est uniformément fixé à 35, leur
    # flux n'a aucun rôle physique dans le transport ; le résidu de
    # pénalisation (amplifié par K_mer) y est du bruit -> on l'annule.
    # NB : ce n'est PAS l'ancien hack — les faces d'INTERFACE gardent
    # leur flux physique exact, c'est lui qui pilote l'échange.
    qx[faces_sea_int] = 0.0
    qz[faces_sea_int] = 0.0
    return qx, qz, buoy


# Test statique avant la boucle
print("\n=== Test statique (état initial) ===")
qx, qz, buoy = solve_flow()
qn_sea = np.sqrt(qx**2 + qz**2)[sea1 & sea2 & ~ext]
print(f"|q| max DANS la mer      : {qn_sea.max():.3e} m/j "
      f"(avant correction : ~2.5e4 ; attendu ~ vitesse physique)")
tmp = FaceVariable(mesh=mesh, rank=1, value=np.array([qx, qz]))
div_q = np.array(tmp.divergence)
div_aq = np.abs(div_q[mask_aq & ~mask_mer]).max()
print(f"div(q) max aquifère      : {div_aq:.3e} 1/j (attendu ~ 0 machine)")
# flux entrant à gauche
qn_left = qx[faces_left]
print(f"Inflow gauche            : {qn_left.sum()*dz:.3f} m2/j "
      f"(imposé Qf = {Qf})")

# ------------------------------------------------------------------
# Boucle temporelle
# ------------------------------------------------------------------
X2D, Z2D = x_cells.reshape(Nz, Nx), z_cells.reshape(Nz, Nx)
mask_plot = (mask_mer | mask_sec).reshape(Nz, Nx)
mc1, mc2 = c1 >= 0, c2 >= 0
frame_index = 0

def to_cell(vf):
    vc = np.zeros(Nx*Nz); cc = np.zeros(Nx*Nz)
    np.add.at(vc, c1[mc1], vf[mc1]); np.add.at(vc, c2[mc2], vf[mc2])
    np.add.at(cc, c1[mc1], 1);       np.add.at(cc, c2[mc2], 1)
    return vc/np.maximum(cc, 1)

def plot_frame(it, t, qx, qz):
    global frame_index
    C2 = np.ma.masked_where(mask_plot, C.value.reshape(Nz, Nx))
    vx2 = np.where(mask_plot, np.nan, to_cell(qx).reshape(Nz, Nx))
    vz2 = np.where(mask_plot, np.nan, to_cell(qz).reshape(Nz, Nx))
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    # Panneau 1 : salinité + LIGNES DE COURANT (visualise la recirculation)
    cf = ax[0].contourf(X2D, Z2D, C2, levels=np.linspace(0, 35, 21),
                        cmap='RdYlBu_r', extend='both')
    plt.colorbar(cf, ax=ax[0], label='C [ppt]')
    ax[0].streamplot(X2D, Z2D, vx2, vz2, color='k', density=1.3,
                     linewidth=0.7, arrowsize=1.1)
    ax[0].set_title(f'Salinité + lignes de courant — t={t:.0f} j  '
                    f'[{C.value[mask_aq].min():.2f}, '
                    f'{C.value[mask_aq].max():.2f}]')
    # Panneau 2 : norme du flux + flèches de direction
    nv = np.sqrt(np.nan_to_num(vx2)**2 + np.nan_to_num(vz2)**2)
    nv = np.ma.masked_where(mask_plot, nv)
    cf2 = ax[1].contourf(X2D, Z2D, nv, levels=20, cmap='Blues')
    plt.colorbar(cf2, ax=ax[1], label='|q| [m/j]')
    sk = 4
    nq = nv[::sk, ::sk] + 1e-12
    ax[1].quiver(X2D[::sk, ::sk], Z2D[::sk, ::sk],
                 np.nan_to_num(vx2)[::sk, ::sk]/nq,
                 np.nan_to_num(vz2)[::sk, ::sk]/nq,
                 color='k', scale=35, width=0.002)
    ax[1].set_title('Flux de Darcy (aquifère)')
    xs = np.linspace(-A/tan_beta, A/tan_beta, 50)
    for a in ax:
        a.plot(xs, -tan_beta*xs, 'k-', lw=2)
        a.axhline(0, color='b', ls='--', lw=1)
        a.set_xlim(-60, 50); a.set_ylim(-H, A)
        a.set_xlabel('x [m]'); a.set_ylabel('z [m]')
    plt.tight_layout()
    plt.savefig(OUT/f"frame_{frame_index:05d}.png", dpi=110)
    plt.close(fig)
    frame_index += 1

# ------------------------------------------------------------------
# Paramètres du run long
# ------------------------------------------------------------------
WALL_MAX  = None       # limite de temps de calcul en secondes (None = aucune)
RESUME    = True       # reprendre depuis le checkpoint s'il existe
CHECKPOINT = OUT / "checkpoint.pkl"

t, dt_max, n_steps = 0.0, 2.0, 10**9
t_end   = 3000.0                     # jours simulés (quasi-stationnaire)
t0_wall = walltime.time()
import pickle
if RESUME and CHECKPOINT.exists():
    st = pickle.load(open(CHECKPOINT, "rb"))
    C.setValue(st["C"]); C.updateOld()
    t, frame_index = st["t"], st["frame_index"]
    print(f"=== REPRISE depuis le checkpoint : t = {t:.1f} j ===")
    print("    (pour repartir de zéro : supprimer checkpoint.pkl "
          "du dossier de frames)")
print("\n=== Simulation (Model II : pas de marée) ===")
IN_hist, M_hist = [], []

for it in range(n_steps+1):
    qx, qz, buoy = solve_flow()
    q_face.setValue(np.array([qx, qz]))

    qn = np.sqrt(qx**2 + qz**2) + 1e-30
    Da = np.zeros((2, 2, nF))
    Da[0, 0] = alpha_L*qx**2/qn + alpha_T*qz**2/qn + ne_aq*Dm
    Da[1, 1] = alpha_L*qz**2/qn + alpha_T*qx**2/qn + ne_aq*Dm
    Da[0, 1] = Da[1, 0] = (alpha_L - alpha_T)*qx*qz/qn
    D_face.setValue(Da)

    vmax = max(np.abs(qx).max()/dx, np.abs(qz).max()/dz)/ne_aq + 1e-30
    dt = min(0.3/vmax, dt_max)

    C.updateOld()
    eq_C.solve(var=C, dt=dt, solver=solver)
    C.setValue(np.where(mask_mer, C_mer, C.value))     # mer réimposée
    t += dt

    if it % 150 == 0:
        tmp = FaceVariable(mesh=mesh, rank=1, value=np.array([qx, qz]))
        div_aq = np.abs(np.array(tmp.divergence)[mask_aq]).max()
        # échange à l'interface mer/aquifère
        qn_int = (face_normals[0]*qx + face_normals[1]*qz)*sign_into_aq
        A_int  = np.where(np.abs(face_normals[1]) > 0.5, dx, dz)
        flux_in  =  np.sum(np.maximum(qn_int[faces_interface], 0)
                           * A_int[faces_interface])
        flux_out = -np.sum(np.minimum(qn_int[faces_interface], 0)
                           * A_int[faces_interface])
        Caq = C.value[mask_aq]
        M_sel = ne_aq*np.sum(np.maximum(Caq-C_terre, 0))*dx*dz
        # position du pied du biseau (C>17.5 la plus à gauche, bas aquifère)
        toe = x_cells[(C.value > C_mer/2) & mask_aq & (z_cells < -H+3*dz)]
        toe_x = toe.min() if toe.size else np.nan
        plot_frame(it, t, qx, qz)
        pickle.dump(dict(C=C.value.copy(), t=t, frame_index=frame_index),
                    open(CHECKPOINT, "wb"))
        print(f"it={it:6d} | t={t:6.1f} j | dt={dt:.3f} | "
              f"C_aq=[{Caq.min():6.3f},{Caq.max():6.3f}] | "
              f"div(q)={div_aq:.1e} | IN={flux_in:.3f} OUT={flux_out:.3f} "
              f"m2/j | M_sel={M_sel:8.1f} | pied biseau x={toe_x:6.1f} m")
        # --- critère d'arrêt physique : IN et M_sel stationnaires ---
        IN_hist.append(flux_in); M_hist.append(M_sel)
        if len(M_hist) > 20:
            dIN = abs(IN_hist[-1]-IN_hist[-11])/max(IN_hist[-1], 1e-12)
            dM  = abs(M_hist[-1]-M_hist[-11])/max(M_hist[-1], 1e-12)
            if dIN < 1e-3 and dM < 1e-3:
                print(f">>> QUASI-STATIONNAIRE atteint à t={t:.0f} j "
                      f"(IN={flux_in:.3f} m2/j, M_sel={M_sel:.0f} ; "
                      f"article : IN=0.15, M00=14710)")
                plot_frame(it, t, qx, qz)
                break
    if t >= t_end or (WALL_MAX is not None
                      and walltime.time()-t0_wall > WALL_MAX):
        plot_frame(it, t, qx, qz)
        print(f"Arrêt à t={t:.1f} j (it={it})")
        break

print("fin")